# Entrenar un LLM desde cero en Google Colab

Notebook basado en el repo [FareedKhan-dev/train-llm-from-scratch](https://github.com/FareedKhan-dev/train-llm-from-scratch).

Ese repo construye un modelo tipo GPT **pieza a pieza en PyTorch puro** y lo lleva por todas las fases que siguen los modelos de verdad:

1. **Preentrenamiento**: el modelo lee mucho texto (The Pile) y aprende a predecir la siguiente palabra.
2. **SFT**: le enseñamos a contestar en formato de chat con ejemplos de pregunta y respuesta.
3. **DPO** (opcional): le enseñamos a preferir respuestas buenas frente a malas.
4. **GRPO** (opcional): aprendizaje por refuerzo con problemas de matemáticas (como DeepSeek-R1).
5. **Evaluación** en problemas de matemáticas (GSM8K) y **chat** con el modelo.

**Qué esperar:** vamos a entrenar un modelo de **~77 millones de parámetros** (unas 2.000 veces más pequeño que GPT-3).
Escribirá frases con palabras reales y gramática más o menos correcta, pero **no será útil de verdad**:
se inventará cosas y fallará casi todos los problemas de matemáticas. El objetivo es **aprender cómo funciona**, no conseguir un buen modelo.

**GPU:** Colab Pro con **A100** (lo ideal) o **L4**. Con T4 funciona, pero usa un modelo más pequeño (se ajusta solo en la celda de configuración).

**Tiempos aproximados** (estimados, no medidos en Colab):

| Fase | A100 | L4 |
|---|---|---|
| Descargar y preparar datos | 10–20 min | 10–20 min |
| Preentrenamiento (2.000 pasos, ~200M tokens) | ~40 min | ~1,5 h |
| SFT | ~10 min | ~20 min |
| DPO (opcional) | ~15 min | ~30 min |
| GRPO (opcional) | ~30 min | ~1 h |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Configuración

Si Colab se desconecta, se pierde todo lo que hay en la máquina. Con `GUARDAR_EN_DRIVE = True`
los modelos entrenados se copian a tu Google Drive al final de cada fase.

In [ ]:
GUARDAR_EN_DRIVE = True
CARPETA_DRIVE = "/content/drive/MyDrive/train-llm-from-scratch"

# Tamaño del modelo y del entrenamiento
PASOS_PREENTRENAMIENTO = 2000
TOKENS_DE_ENTRENAMIENTO = 250_000_000   # cuántos tokens de The Pile preparar

import torch, subprocess
gpu = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
soporta_bf16 = torch.cuda.is_bf16_supported()

if mem_gb >= 35:        # A100 40/80 GB
    MODELO = dict(n_embed=512, n_head=8, n_blocks=8, context_length=512)
    BATCH, ACUMULAR = 24, 8
elif mem_gb >= 20:      # L4 24 GB
    MODELO = dict(n_embed=512, n_head=8, n_blocks=8, context_length=512)
    BATCH, ACUMULAR = 12, 16
else:                   # T4 16 GB: modelo más pequeño
    MODELO = dict(n_embed=384, n_head=6, n_blocks=6, context_length=256)
    BATCH, ACUMULAR = 16, 12

CTX = MODELO["context_length"]
print(f"GPU: {gpu} ({mem_gb:.0f} GB) | bf16: {soporta_bf16}")
print(f"Modelo: {MODELO} | lote efectivo = {BATCH} x {ACUMULAR} = {BATCH*ACUMULAR} secuencias por paso")

if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 2. Descargar el repo e instalarlo

Fijamos una versión concreta del repo para que el notebook no se rompa si el autor cambia algo.
El repo guarda todo en `/ephemeral` (datos, modelos y registros), así que usamos esa misma carpeta.

In [ ]:
!git clone -q https://github.com/FareedKhan-dev/train-llm-from-scratch.git /content/train-llm-from-scratch
%cd /content/train-llm-from-scratch
!git checkout -q b995104cff1dd488cdcc1517bc284649d9252d88
!pip install -q -e ".[train]"
!mkdir -p /ephemeral/data /ephemeral/ckpts /ephemeral/logs /ephemeral/hf_cache

import os
os.environ["HF_HOME"] = "/ephemeral/hf_cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Pruebas rápidas del propio repo: comprueban que las matemáticas del código están bien (tarda segundos)
!python tests/test_post_training_smoke.py

## 3. Preparar los datos

Usamos **The Pile** (versión sin material con copyright), un conjunto enorme de texto en inglés de webs, libros, artículos científicos, código…

- La parte de validación pesa unos 340 MB.
- Para entrenar se descarga un trozo de unos **11 GB**, pero solo convertimos a tokens lo que vamos a usar.

"Tokenizar" es convertir el texto en números: cada trozo de palabra pasa a ser un número. Es lo único que el modelo entiende.

In [ ]:
!python scripts/prepare_pretrain_data.py --split val --raw_dir /ephemeral/data/pile_raw --out /ephemeral/data/pile_dev.h5 --max_tokens 5000000
!python scripts/prepare_pretrain_data.py --split train --num_shards 1 --raw_dir /ephemeral/data/pile_raw --out /ephemeral/data/pile_train.h5 --max_tokens {TOKENS_DE_ENTRENAMIENTO}

# Ya no hacen falta los ficheros comprimidos originales
!rm -rf /ephemeral/data/pile_raw
!du -sh /ephemeral/data/*

## 4. Configurar el modelo

El repo lee la configuración de ficheros JSON. Creamos los nuestros en `configs/colab/` con un tamaño que quepa en una sola GPU.

In [ ]:
import json, os
os.makedirs("configs/colab", exist_ok=True)

def guardar(nombre, datos):
    with open(f"configs/colab/{nombre}.json", "w") as f:
        json.dump(datos, f, indent=2)

guardar("base", dict(vocab_size=50304, **MODELO, device="cuda",
                     amp_dtype="bf16" if soporta_bf16 else None, seed=1337, compile=False,
                     ckpt_dir="/ephemeral/ckpts", log_dir="/ephemeral/logs", use_wandb=False))

guardar("pretrain", dict(train_path="/ephemeral/data/pile_train.h5", dev_path="/ephemeral/data/pile_dev.h5",
                         batch_size=BATCH, grad_accum=ACUMULAR, train_steps=PASOS_PREENTRENAMIENTO,
                         eval_steps=250, eval_iters=20, warmup_steps=100, lr=6e-4, min_lr=6e-5,
                         weight_decay=0.1, grad_clip=1.0, save_every=500,
                         out_ckpt="/ephemeral/ckpts/base_pretrained.pt"))

guardar("sft", dict(pretrained_ckpt="/ephemeral/ckpts/base_pretrained.pt",
                    data_path="/ephemeral/data/sft_packed.h5", out_ckpt="/ephemeral/ckpts/sft.pt",
                    batch_size=16, epochs=2, eval_steps=500, warmup_steps=100,
                    lr=5e-5, min_lr=5e-6, save_every=100000))

guardar("dpo", dict(sft_ckpt="/ephemeral/ckpts/sft.pt", pref_path="/ephemeral/data/preferences.jsonl",
                    out_ckpt="/ephemeral/ckpts/dpo.pt", loss_type="dpo", beta=0.1,
                    batch_size=8, epochs=1, eval_steps=500, warmup_steps=50, lr=1e-6,
                    max_len=MODELO["context_length"], save_every=100000))

guardar("grpo", dict(sft_ckpt="/ephemeral/ckpts/sft.pt",
                     prompt_path="/ephemeral/data/rl_prompts_train.jsonl",
                     eval_prompt_path="/ephemeral/data/rl_prompts_test.jsonl",
                     curriculum_path="/ephemeral/data/arithmetic_prompts.jsonl",
                     curriculum_iters=50, out_ckpt="/ephemeral/ckpts/grpo.pt",
                     iterations=200, prompts_per_iter=8, group_size=8,
                     rollout_len=min(200, MODELO["context_length"] // 2),
                     lr=2e-6, kl_coef=0.04, eval_every=50, save_every=100))

from src.models.transformer import Transformer
m = Transformer(n_head=MODELO["n_head"], n_embed=MODELO["n_embed"], context_length=MODELO["context_length"],
                vocab_size=50304, N_BLOCKS=MODELO["n_blocks"])
print(f"Parámetros del modelo: {sum(p.numel() for p in m.parameters())/1e6:.0f} millones")
del m

In [ ]:
import shutil

def guardar_en_drive(*ficheros):
    if not GUARDAR_EN_DRIVE:
        return
    os.makedirs(CARPETA_DRIVE, exist_ok=True)
    for f in ficheros:
        if os.path.exists(f):
            shutil.copy(f, CARPETA_DRIVE)
            print("copiado a Drive:", f)

## 5. Preentrenamiento

Es la fase larga. El modelo lee trozos de texto y en cada posición intenta adivinar la siguiente palabra.
Cuando falla, se corrige un poco. Así miles de veces.

El número que hay que mirar es la **pérdida** (*loss*): cuanto más baja, mejor adivina.
Empieza cerca de **10,8** (adivinar al azar entre 50.000 tokens) y en el repo original baja hasta **~3,7** en 2.000 pasos.

Si Colab se corta a mitad, se puede continuar desde el último punto guardado añadiendo
`--resume /ephemeral/ckpts/base_pretrained.pt` (el fichero tiene que seguir en la máquina o copiarse desde Drive).

In [ ]:
!python scripts/pretrain_base.py --config configs/colab/pretrain.json
guardar_en_drive("/ephemeral/ckpts/base_pretrained.pt")

In [ ]:
# Gráfica de la pérdida
import glob, json
import matplotlib.pyplot as plt

log = sorted(glob.glob("/ephemeral/logs/pretrain_*.jsonl"))[-1]
filas = [json.loads(l) for l in open(log)]
train = [(r["step"], r["train_loss"]) for r in filas if "train_loss" in r]
dev = [(r["step"], r["eval_dev"]) for r in filas if "eval_dev" in r]

plt.figure(figsize=(8, 4))
plt.plot(*zip(*train), label="entrenamiento", alpha=0.7)
if dev:
    plt.plot(*zip(*dev), "o-", label="validación (texto no visto)")
plt.axhline(10.83, ls="--", c="gray", label="adivinar al azar")
plt.xlabel("paso"); plt.ylabel("pérdida"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

### Probar el modelo base

Un modelo recién preentrenado **no sabe conversar**: solo sabe continuar texto.

In [ ]:
!python scripts/chat.py --ckpt /ephemeral/ckpts/base_pretrained.pt --raw --prompt "The history of the Roman Empire" --max_new_tokens 80
!python scripts/chat.py --ckpt /ephemeral/ckpts/base_pretrained.pt --raw --prompt "def fibonacci(n):" --max_new_tokens 60

## 6. SFT: enseñarle a contestar

Usamos ~75.000 ejemplos de pregunta y respuesta (Alpaca, Dolly y problemas de matemáticas GSM8K).
Sigue siendo "adivinar la siguiente palabra", pero solo se le corrige en la parte de la **respuesta**, no en la pregunta.

En los problemas de matemáticas aprende a contestar con este formato:
`<think>razonamiento</think><answer>número</answer>`.

In [ ]:
!python scripts/prepare_sft_data.py --context_length {CTX} --out_dir /ephemeral/data
!python scripts/train_sft.py --config configs/colab/sft.json
guardar_en_drive("/ephemeral/ckpts/sft.pt")

In [ ]:
!python scripts/chat.py --ckpt /ephemeral/ckpts/sft.pt --prompt "What is the capital of France?" --max_new_tokens 60
!python scripts/chat.py --ckpt /ephemeral/ckpts/sft.pt --prompt "Give me three tips to sleep better." --max_new_tokens 100
!python scripts/chat.py --ckpt /ephemeral/ckpts/sft.pt --prompt "What is 13 + 29?" --max_new_tokens 60 --greedy

## 7. (Opcional) DPO: preferir respuestas buenas

Le damos parejas de respuestas a la misma pregunta, una buena y una mala (escogidas por personas),
y le enseñamos a dar más probabilidad a la buena. Es la técnica que usan muchos modelos actuales en lugar del RLHF clásico.

In [ ]:
!python scripts/prepare_preference_data.py --source both --max_per_source 10000 --out_dir /ephemeral/data
!python scripts/train_dpo.py --config configs/colab/dpo.json
guardar_en_drive("/ephemeral/ckpts/dpo.pt")

## 8. (Opcional) GRPO: aprendizaje por refuerzo con matemáticas

El modelo genera varias respuestas a cada problema. Las que aciertan el número final reciben premio y las demás no.
Primero practica con sumas y restas sencillas y luego pasa a GSM8K.

Con un modelo tan pequeño lo normal es que **casi nunca acierte**, así que apenas recibe premios y aprende muy poco.
Sirve para ver cómo funciona el método, no para mejorar mucho el modelo.

In [ ]:
!python scripts/prepare_rl_prompts.py --out_dir /ephemeral/data
!python scripts/train_grpo.py --config configs/colab/grpo.json
guardar_en_drive("/ephemeral/ckpts/grpo.pt")

## 9. Evaluación: problemas de matemáticas (GSM8K)

Mide el porcentaje de problemas de matemáticas de primaria que resuelve cada versión del modelo.
Con ~77M parámetros es normal sacar **0–2 %**. Como referencia, los modelos de 7–8 mil millones de parámetros sacan 50–90 %.

In [ ]:
!rm -f /ephemeral/logs/stage_table.jsonl
for etapa in ["base_pretrained", "sft", "dpo", "grpo"]:
    ckpt = f"/ephemeral/ckpts/{etapa}.pt"
    if os.path.exists(ckpt):
        !python scripts/eval_post_training.py --ckpt {ckpt} --label {etapa} --limit 200 --max_new_tokens 200 --samples 1 --append /ephemeral/logs/stage_table.jsonl
!python scripts/eval_post_training.py --table /ephemeral/logs/stage_table.jsonl

## 10. Hablar con el modelo

Cambia el fichero por el de la fase que quieras probar (`sft.pt`, `dpo.pt` o `grpo.pt`) y el texto de la pregunta.
El modelo solo ha visto inglés, así que pregúntale en inglés.

In [ ]:
PREGUNTA = "Explain what a neural network is in one sentence."
!python scripts/chat.py --ckpt /ephemeral/ckpts/sft.pt --prompt "{PREGUNTA}" --max_new_tokens 120 --temperature 0.7